# To auto-label NER trainset with Chat Model.

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 
from dotenv import load_dotenv
import ast
import pandas as pd

load_dotenv()

chat = ChatOpenAI(
    model_name='gpt-4-turbo-preview',
    temperature=0,
    streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
    )

examples1 = [
    {
        'text': '발진이안나네요',
        'subject_keywords': '''['발진']'''
        
    },
    {
        'text': '이상품은 발진이 안나요. 좋아요',
        'subject_keywords': '''['상품', '발진']'''
    },
    {
        'text': '사이즈가 너무 커요.',
        'subject_keywords': '''['사이즈']'''
    },
    {
        'text': '이번 제품은 괜찮고, 깨끗해요.',
        'subject_keywords': '''['제품']'''
    },
    {
        'text': '리뷰 싫어!!',
        'subject_keywords': '''['-']'''
    },
]

# example_template = '''
#     Human: {question}
#     AI: {answer}
# '''

example1_prompt = ChatPromptTemplate.from_messages([
    ('human', '{text}'),
    ('ai', '{subject_keywords}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example1_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example1_prompt,
    examples=examples1
)

stage1_prompt = ChatPromptTemplate.from_messages([
    ('system', """you're data scientist. From now on, you'll find all nouns in the text human says in sequential order. 
            Do not modify the found noun texts in the text.
            If you cannot find any subject keywords, just return '-' as a type of list. 
            You give short answers."""),
    example1_prompt,
    ('human', '{text}')
])

##################################################################################################################################

examples2 = [
    {
        'human': '''
            text: 발진이안나네요
            keywords: ['발진']
        ''',
        'answer': '''{'발진': [('안나네요', 'positive')]}'''
        
    },
    {
        'human': '''
            text: 이상품은 발진이 안나요. 좋아요
            keywords: ['상품', '발진']
        ''',
        'answer': '''{'상품': [('좋아요', 'positive')], '발진': [('안나요', 'positive')]}'''
        
    },
    {
        'human': '''
            text: 사이즈가 너무 커요.
            keywords: ['사이즈']
        ''',
        'answer': '''{'사이즈': [('너무 커요', 'negative')]}'''
        
    },
    {
        'human': '''
            text: 이번 제품은 괜찮고, 깨끗해요.
            keywords: ['제품']
        ''',
        'answer': '''{'제품': [('괜찮고', 'positive'), ('깨끗해요', 'positive')]}'''
        
    },
     {
        'human': '''
            text: 리뷰 싫어!!
            keywords: ['-']
        ''',
        'answer': '''{'-': [('-', '-')]}'''
        
    },
]

# example_template = '''
#     Human: {question}
#     AI: {answer}
# '''

example2_prompt = ChatPromptTemplate.from_messages([
    ('human', '{human}'),
    ('ai', '{answer}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example2_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example2_prompt,
    examples=examples2
)

stage2_prompt = ChatPromptTemplate.from_messages([
    ('system', """A review text and the subject keywords it contains will be given. 
            You'll find all sentiment keywords corresponding with the given subject keywords in the text. 
            Find all sentiment keywords and tag a sentiment(positive/negative/neutral) as a form of tuple for each keywords given, and put together them as a list.
            After finding all sentimental keywords for every given keywords, put together the results as a form of Dict().
            If you cannot find any words and sentiments for each word, you can use '-' instead.
            DO NOT modify the found words in the text.
            Keep the output format throughly."""),
    example2_prompt,
    ('human', '{human}')
])

##################################################################################################################################

samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… 잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

st1_chain = stage1_prompt | chat
st2_chain = stage2_prompt | chat

final_df = pd.DataFrame(columns=['text_no', 'subject_word', 'sentiment_word', 'sentiment_val', 'texts'])
for text_idx, sample in enumerate(samples, start=1):
    st1_answer = st1_chain.invoke({
        'text': sample
    })
    # print(st1_answer.content)
    st1_list = st1_answer.content
    # print(st1_list)

    st2_answer = st2_chain.invoke({
        'human': 'text: {}\nkeywords: {}'.format(sample, st1_list)
    })
    print(st2_answer.content)

    st2_dict = ast.literal_eval(st2_answer.content)
    for key, val_list in st2_dict.items():
        size = len(val_list)
        temp_dict = {'text_no': [text_idx] * size, 'subject_word': [key] * size, 'sentiment_word': [x[0] for x in val_list], 'sentiment_val': [x[1] for x in val_list], 'texts': [sample] * size}
        temp_df = pd.DataFrame.from_dict(temp_dict)
        final_df = pd.concat([final_df, temp_df], axis=0)

    final_df.to_csv('chatgpt4.0_sample_result.csv', encoding='utf-8-sig', index=False)


    

Authentication failed for https://api.smith.langchain.com/runs. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs', '{"error":"Unauthorized: Using outdated v1 api key. Please use v2 api key."}\n')
Authentication failed for https://api.smith.langchain.com/runs/4cb3478a-6c1e-4dce-9f5b-09b0b4ead0fc. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/4cb3478a-6c1e-4dce-9f5b-09b0b4ead0fc', '{"detail":"Using legacy API key. Please generate a new API key."}')


{'-': [('-', '-')]}


KeyboardInterrupt: 

In [ ]:
##### 1Q Ver. TEST #####

from langchain.chat_models import ChatOpenAI
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler, get_openai_callback 
# from langchain.globals import set_llm_cache, set_debug
# from langchain.cache import InMemoryCache
from dotenv import load_dotenv
import ast
import pandas as pd

load_dotenv()


# set_llm_cache(InMemoryCache())
# set_debug(True)

chat = ChatOpenAI(
    model_name='gpt-4-turbo-preview',
    temperature=0,
    streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
    )

examples = [
    {
        'text': '발진이안나네요',
        'answer': '''{"발진": [("안나네요", "positive")]}'''
        
    },
    {
        'text': '이상품은 발진이 안나요. 좋아요',
        'answer': '''{"상품": [("좋아요", "positive")], "발진": [("안나요", "positive")]}'''
        
    },
    {
        'text': '사이즈가 너무 커요.',
        'answer': '''{"사이즈": [("너무 커요", "negative")]}'''
    },
    {
        'text': '이번 제품은 괜찮고, 깨끗해요.',
        'answer': '''{"제품": [("괜찮고", "positive"), ("깨끗해요", "positive")]}'''
    },
    {
        'text': '리뷰 싫어!!',
        'answer': '''{"-": [("-", "-")]}'''
    },
]

intro = PromptTemplate.from_template(
    """
    You're data labeller for NER Model. At first, you'll find all nouns in the text human says in sequential order. 
    Do not modify the found noun texts in the text.
    I'm going to call them SUBJECT KEYWORDS.
    If you cannot find any subject keywords, just return '-' as a type of list. 
   
    Then, you'll find all sentiment keywords and tag a sentiment(positive/negative/neutral) for each subject keywords.
    If you cannot find any words and sentiments for each word, you can use '-' instead.
    Again, DO NOT modify the found words in the text.

    Final output format will be string of Dict() type, which has subject keywords as keys, 
    and a list of tuples (sentiment keywords, sentiment) as its values.
    Keep the output format below throughly.
"""
)

example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{text}'),
    ('ai', '{answer}')
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {examples}

    {start}
"""
)

prompts = [
    ("intro", intro),
    ("examples", example_prompt),
    ("start", start),
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

# full_prompt.format(
#     character='Pirate',
#     example_question='What is your location?',
#     example_answer='Arrrg! That is a secret!! Arg Arg!!',
#     question='What is your fav food?',
# )

chain = full_prompt | chat

samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… 잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

# with get_openai_callback() as usage:
final_df = pd.DataFrame(columns=['text_no', 'subject_word', 'sentiment_word', 'sentiment_val', 'texts'])
for text_idx, sample in enumerate(samples, start=1):
    answer = chain.invoke(
        {
            "question": sample
        }
    )
    print(answer.content)

    rslt_dict = ast.literal_eval(answer.content)
    for key, val_list in rslt_dict.items():
        size = len(val_list)
        temp_dict = {'text_no': [text_idx] * size, 'subject_word': [key] * size, 'sentiment_word': [x[0] for x in val_list], 'sentiment_val': [x[1] for x in val_list], 'texts': [sample] * size}
        temp_df = pd.DataFrame.from_dict(temp_dict)
        final_df = pd.concat([final_df, temp_df], axis=0)

final_df.to_csv('chatgpt4.0_sample_result_v2.csv', encoding='utf-8-sig', index=False)
        # print(usage)

{"-": [("너무 좋아요~", "positive")]}
{"제품": [("너무 좋아요", "positive")]}
{"기저귀값": [("두배", "negative"), ("급격한", "negative"), ("가격변화", "negative")], "기저귀": [("찾아볼까", "neutral")]}
{"밴드": [("부드러워서", "positive"), ("좋아요", "positive")], "샘방지포켓": [("없어서", "negative")], "재구매의사": [("있어요", "positive")]}
{"출산": [("내년인데", "neutral"), ("쌍둥이 출산예정이라", "neutral")], "멤버쉽": [("네이버", "neutral")], "행사": [("있고", "positive")], "세일": [("있고", "positive")], "손수건": [("이것저것", "neutral"), ("종류별로", "neutral")], "기저귀": [("목욕타월,", "neutral"), ("낮잠 이불", "neutral")], "디자인": [("확실히 이쁘네요", "positive")], "엄마": [("엄마취향으로다", "positive")]}
{"하기스": [("잘샌다고", "positive"), ("괜찮은것", "positive")], "밴딩": [("괜찮은것", "positive")], "두께": [("얇아요", "positive")], "흡수력": [("괜찮고", "positive")], "통풍": [("잘 될것", "positive")], "발진": [("안보이네요", "positive")], "팩": [("-", "-")]}
{"배송": [("빠르고", "positive"), ("좋네요", "positive")], "물놀이": [("-", "-")], "저렴했어요": [("저렴했어요", "positive")], "기저귀": [("일반", "neutral"), ("별로 다른게 없어 보였눈데", "neutral"), ("방수력", "pos

In [ ]:
##### 1Q Ver-2. TEST #####

from langchain.chat_models import ChatOpenAI
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler, get_openai_callback 
# from langchain.globals import set_llm_cache, set_debug
# from langchain.cache import InMemoryCache
from dotenv import load_dotenv
import ast
import pandas as pd
import re
import os

from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough


cache_dir = LocalFileStore("./.cache/")

load_dotenv()

# set_llm_cache(InMemoryCache())
# set_debug(True)

chat = ChatOpenAI(
        model_name='gpt-4-turbo-preview',
        temperature=0,
        streaming=True,
        # callbacks=[
        #     StreamingStdOutCallbackHandler(),
        # ],
    )

instruction = PromptTemplate.from_template(
    """
    You're a data labeller for NER Model. Here is what you should do.

    Each paragraph of the following context starts with index. (e.g. [1], [2], [3]...)
    You'll find subject keywords and its sentiments for each paragraph according to the instruction below.

    At first, you'll find all nouns in sequential order throughout a paragraph. 
    Do not modify the found noun texts in the text.
    I'm going to call them SUBJECT KEYWORDS.
    If you cannot find any subject keywords, just return '-'. 
   
    Then, you'll find all sentiment keywords and tag a sentiment(positive/negative/neutral) for each subject keywords in the document.
    If the subject keyword is '-', find all sentiment keywords and tag a sentiment(positive/negative/neutral) through the document.
    If you cannot find any words and sentiments for each word, you can use '-' instead.
    Again, DO NOT modify the found words in the text.

    Output format will be like the example below.
    Keep the output format below throughly.
"""
)

examples = ChatPromptTemplate.from_messages([
    ('human', '''Example Context: 
                [0] 발진이안나네요 [1] 이상품은 발진이 안나요. 좋아요 [2] 사이즈가 너무 커요. [3] 이번 제품은 괜찮고, 깨끗해요. [4] 리뷰 싫어!!'''),
    ('ai', '''
            0: (발진, 안나네요, positive)
            1: (상품, 좋아요, positive), (발진, 안나요, positive)
            2: (사이즈, 너무 커요, negative)
            3: (제품, 괜찮고, positive), (제품, 깨끗해요, positive)
            4: (-, -, -)
        ''')
])

# examples = PromptTemplate.from_template(
#     '''
#     This is an example of context and what you should return:

#     Example Context: 
#     [0] 발진이안나네요 [1] 이상품은 발진이 안나요. 좋아요 [2] 사이즈가 너무 커요. [3] 이번 제품은 괜찮고, 깨끗해요. [4] 리뷰 싫어!!

#     You: 

# '''
# )
# [{"발진": [("안나네요", "positive")]}, {"상품": [("좋아요", "positive")], "발진": [("안나요", "positive")]}, {"사이즈": [("너무 커요", "negative")]}, {"제품": [("괜찮고", "positive"), ("깨끗해요", "positive")]}, {"-": [("-", "-")]}]


samples = [
    '너무 좋아요~',
    '이 제품 너무 좋아요!!',
    '아이에게 잘맞아서 자주 주문했는데… \n\n잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.',
    '밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요',
    '출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ',
    '애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~',
    '배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ'
]

samples = ['[{}] '.format(idx) + re.sub('\n', ' ', sample) for idx, sample in enumerate(samples)]

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n\n',
)

docs = splitter.create_documents(samples)
print(len(docs), docs)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriver = vectorstore.as_retriever()

documents = PromptTemplate.from_template(
    """
    Here's the context:\n\n{context}
"""
)

final = PromptTemplate.from_template(
    """
    {instruction}
                                     
    {examples}

    {documents}

    {start}
"""
)

prompts = [
    ("instruction", instruction),
    ("examples", examples),
    ("documents", documents)
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

chain = (
    {
        "context": retriver,
        "start": RunnablePassthrough(),
    }
    | full_prompt
    | chat
)

response = chain.invoke("Follow the instruction above and return output!")
print(response.content)


7 [Document(page_content='[0] 너무 좋아요~'), Document(page_content='[1] 이 제품 너무 좋아요!!'), Document(page_content='[2] 아이에게 잘맞아서 자주 주문했는데…   잠깐 사이에 기저귀값이 두배?정도 뛴듯해요.너무 급격한 가격변화라 다른 기저귀를 찾아볼까 고민중입니다.'), Document(page_content='[3] 밴드가 부드러워서 좋아요샘방지포켓이 없어서 가끔 앞으로 샐때가있기는 했어요 그래도 재구매의사는 있어요'), Document(page_content='[4] 출산 내년인데 네이버 멤버쉽 행사도 있고 세일도 있고!! 미리 구매했어요~ 많으면 많을수록 좋다고 하고 쌍둥이 출산예정이라 이것저것 손수건 종류별로 많이 구매했어요기저귀는 목욕타월, 낮잠 이불 등등 다용도로 사려구 구매했어요~ 저는 디자인이 있는게 확실히 이쁘네요~ 엄마취향으로다 골랐슴다 ㅎㅎ'), Document(page_content='[5] 애기엄마들 사이에서 하기스가 잘샌다고 하던데 잘 몸에 맞춰서 밴딩하면 괜찮은것 같아요~두께는 정말 얇아요~흡수력도 괜찮고 통풍이 잘 될것 같아요쓰고 아직까진 발진은 안보이네요^^처음쓰는거라서 한팩 써보고 더사용할지 고민하려고 해요~'), Document(page_content='[6] 배송 빠르고 좋네요물놀이 하려고 샀어요 ㅎㅎ 다른데 보다 저렴했어요입혀봤는데 진짜 신기하네요 ㅋㅋ 보기에는 일반 기저귀랑 별로 다른게 없어 보였눈데 방수력 !!ㅋㅋ 일반 기저귀보다 얇아요요긴하게 잘 썼습니다 ㅎ')]


KeyError: '발진'